# Step 5 · The next batch

**Goal:** choose the 5 experiments to run next.

| Slots | Role | How it is chosen | Why |
|---|---|---|---|
| 3 | **Exploit** | Highest Expected Improvement | With few, costly experiments, most slots should go where gains are likely |
| 1 | **Explore** | Most uncertain recipe among those predicted at least average (y ≥ 0) | Learn about an unexplored region, without spending the slot on a recipe expected to fail |
| 1 | **Benchmark** | The commercial baseline (DMEM + 10 % FBS) | Measures how much this batch drifted from earlier ones, and lets us report results as "better than today's medium" |

**Avoiding near-copies:** picks are made one at a time. After each pick we *pretend it was already measured* at its predicted value. The model becomes confident there, so the next pick goes somewhere else. The pretend values are only used to choose this batch and are thrown away afterwards. The trick is known as the *kriging believer* (Ginsbourger et al., 2010).

All of this is wrapped in `recommend()` (in `mediaopt/acquisition.py`): fit the GP → generate candidates → pick the batch → add the benchmark.

In [1]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd

from mediaopt.acquisition import random_batch, recommend, score_candidates
from mediaopt.cost import baseline_cost, media_cost
from mediaopt.data import COLS, load_recipes, to_concentration
from mediaopt.model import fit, make_gp

pd.set_option("display.max_columns", None)

In [2]:

df = load_recipes()
CEILING = baseline_cost()
batch, BEST = recommend(df, CEILING)

# how far each recipe is from the nearest recipe already tested (0-1 scale, 14 components)
batch["distance_to_tested"] = [np.linalg.norm(df[COLS].to_numpy() - r, axis=1).min() for r in batch[COLS].to_numpy()]
print(f"bar to beat (model's estimate of best tested recipe): {BEST:.2f}")
batch[["role", "mean", "low90", "high90", "ei", "cost", "distance_to_tested"]].round(2)

bar to beat (model's estimate of best tested recipe): 1.40


,role,mean,low90,high90,ei,cost,distance_to_tested
0,exploit,1.52,0.45,2.59,0.33,7.06,0.30
1,exploit,1.47,0.39,2.56,0.30,6.35,0.43
2,exploit,1.43,0.39,2.46,0.26,8.01,0.37
3,explore,0.01,-1.81,1.82,0.05,5.67,1.64
4,benchmark,-0.30,-2.12,1.52,0.03,8.08,1.65


**Columns**
- `mean`: predicted cell count.
- `low90`, `high90`: the 90 % interval for its true value (mean ± 1.645 sd, as in step 3).
- `ei`: Expected Improvement.
- `cost`: must be ≤ the ceiling.
- `distance_to_tested`: how new the recipe is.

Here the recommended batch:

In [3]:
batch

,T,I,SS,AA,Glu,Gluta,Albu,FBS,H,D,P,Esd,Ethan,Glutath,role,source,mean,sd,ei,low90,high90,cost,distance_to_tested
0,0.481426,0.986384,0.410367,0.000000,0.383135,1.000000,0.000000,0.494248,0.526968,0.000000,1.000000,0.777435,0.272516,0.974585,exploit,local,1.523915,0.650486,0.325948,0.453866,2.593965,7.055843,0.301241
1,0.888240,1.000000,0.423409,0.000000,0.443031,1.000000,0.000000,0.442442,0.528412,0.000000,1.000000,0.868267,0.130461,1.000000,exploit,local,1.474058,0.659901,0.301749,0.388521,2.559595,6.345399,0.429325
2,0.858846,0.829715,0.080579,0.019699,0.453880,0.886560,0.000000,0.561493,0.000000,0.083160,0.989621,1.000000,0.540820,0.680172,exploit,local,1.425505,0.626897,0.262867,0.394260,2.456751,8.009697,0.368545
3,0.125329,0.551146,0.613247,0.047850,0.394629,0.977128,0.807969,0.110177,0.926437,0.780732,0.101750,0.962466,0.061186,0.886733,explore,random,0.006005,1.101198,0.053845,-1.805465,1.817475,5.669059,1.639014
4,0.000000,0.000000,0.000000,0.000000,0.285714,0.283081,0.000000,0.571429,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,benchmark,baseline,-0.302837,1.107394,0.029743,-2.124500,1.518826,8.084059,1.651974


**Compare with the simplest alternative:** 4 random recipes within the ceiling, chosen without a model.

In [4]:
gp = fit(make_gp(), df)
rand = score_candidates(gp, random_batch(df, CEILING), BEST)
rand["distance_to_tested"] = [np.linalg.norm(df[COLS].to_numpy() - r, axis=1).min() for r in rand[COLS].to_numpy()]

new = batch[batch.role != "benchmark"]
pd.DataFrame({
    "model batch (3 exploit + 1 explore)": new[["mean", "ei", "distance_to_tested"]].mean(),
    "random batch": rand[["mean", "ei", "distance_to_tested"]].mean(),
}).T.rename(columns={"mean": "avg predicted growth", "ei": "avg Expected Improvement",
                     "distance_to_tested": "avg distance to tested"}).round(2)

,avg predicted growth,avg Expected Improvement,avg distance to tested
model batch (3 exploit + 1 explore),1.11,0.24,0.68
random batch,-0.38,0.02,1.48


**The recommendation in lab units** (mg/ml; FBS in % v/v):

In [5]:
lab = to_concentration(batch).T
lab.columns = batch.role
lab

role,exploit,exploit,exploit,explore,benchmark
T,0.012517,0.023094,0.022330,0.003259,0.00
I,0.034523,0.035000,0.029040,0.019290,0.00
SS,0.000007,0.000007,0.000001,0.000011,0.00
AA,0.000000,0.000000,0.000172,0.000419,0.00
Glu,6.034380,6.977732,7.148616,6.215411,4.50
Gluta,1.519000,1.519000,1.346684,1.484257,0.43
Albu,0.000000,0.000000,0.000000,1.131157,0.00
FBS,8.649342,7.742730,9.826133,1.928095,10.00
H,0.000009,0.000009,0.000000,0.000016,0.00
D,0.000000,0.000000,0.000058,0.000547,0.00


**Takeaways**
- **Exploit picks** are predicted at about 1.4-1.5, against a bar of 1.40. Their 90 % intervals run from about 0.4 to 2.6: each could land anywhere between an average recipe and the best ever measured.
  - They share a theme: 8-10 % FBS, no albumin, maximum glutamine and insulin, progesterone.
  - They vary transferrin, glucose, FBS level and the other hormones, so each tests a different variation.
  - Two of them are also **13-21 % cheaper** than the baseline.
- **The explore pick is a low-serum recipe** (≈ 2 % FBS) that relies on albumin and hormones instead.
  - It is far from anything tested; the model predicts it as about average, but is very unsure.
  - It is 30 % cheaper. If it works, it opens a much cheaper direction; if not, we've ruled out a region for one experiment.
- **The benchmark is predicted below average and uncertain.** No tested recipe resembles plain DMEM + 10 % FBS. That is another reason to measure it: it anchors the batch's results to today's medium.
- **Compared with a random batch, the model-guided batch is far more promising:** average predicted growth 1.11 vs −0.38, and Expected Improvement 0.24 vs 0.02. Random's only strength is spread, which our explore slot provides in a targeted way.